# 02 — Multi-Agent Workflow Exploration

This notebook walks through the **Supervisor/Worker pattern** implemented in `victor-multi-agent-workflow`.

**What you'll learn:**
1. How the Supervisor decomposes tasks into sub-tasks
2. How Worker agents execute sub-tasks with tool use
3. How Human-in-the-Loop checkpoints work
4. How to call the FastAPI endpoints directly
5. How to stream agent reasoning via SSE


In [ ]:
# Install dependencies
# !pip install anthropic fastapi uvicorn httpx sse-starlette pydantic-settings structlog

import os
import asyncio
import json
import anthropic

# Load your API key
ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', 'your-key-here')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print('Anthropic client ready.')

## Step 1 — Supervisor Task Decomposition

Send a complex task to the Supervisor and see the plan it generates.

In [ ]:
SUPERVISOR_PROMPT = """
You are a Supervisor AI agent. Decompose the user task into sub-tasks for:
- research_worker: web search, fact-finding
- analyst_worker: data analysis, calculations
- writer_worker: summarization, drafts

Respond with JSON: {"subtasks": [{"worker": "...", "task": "..."}]}
"""

complex_task = "Compare the top 3 open-source LLM frameworks (LangChain, LlamaIndex, Haystack) by GitHub stars, license type, and production readiness."

response = client.messages.create(
    model='claude-3-5-sonnet-20241022',
    max_tokens=512,
    system=SUPERVISOR_PROMPT,
    messages=[{'role': 'user', 'content': complex_task}]
)

raw = response.content[0].text
plan = json.loads(raw[raw.find('{'):raw.rfind('}')+1])
print(f'Plan has {len(plan["subtasks"])} sub-tasks:')
for i, t in enumerate(plan['subtasks'], 1):
    print(f'  {i}. [{t["worker"]}] {t["task"]}')

## Step 2 — Worker Agent with Tool Use

Run a Research Worker with web search tool.

In [ ]:
RESEARCH_TOOLS = [
    {
        'name': 'web_search',
        'description': 'Search the web for information.',
        'input_schema': {
            'type': 'object',
            'properties': {'query': {'type': 'string'}},
            'required': ['query']
        }
    }
]

def mock_web_search(query):
    """Replace with a real search API in production."""
    return f"[Mock results for '{query}']: LangChain ~90k stars MIT, LlamaIndex ~35k stars MIT, Haystack ~17k stars Apache 2.0"

messages = [{'role': 'user', 'content': 'Compare GitHub stars for LangChain, LlamaIndex, and Haystack'}]

for turn in range(5):
    r = client.messages.create(
        model='claude-3-haiku-20240307',
        max_tokens=512,
        system='You are a Research Worker. Use tools to find information.',
        tools=RESEARCH_TOOLS,
        messages=messages
    )
    if r.stop_reason == 'end_turn':
        print('Research Worker output:')
        print(r.content[0].text)
        break
    elif r.stop_reason == 'tool_use':
        tool_results = []
        for block in r.content:
            if block.type == 'tool_use':
                print(f'Tool call: {block.name}({block.input})')
                result = mock_web_search(block.input.get('query', ''))
                tool_results.append({'type': 'tool_result', 'tool_use_id': block.id, 'content': result})
        messages.append({'role': 'assistant', 'content': r.content})
        messages.append({'role': 'user', 'content': tool_results})

## Step 3 — Calling the FastAPI Endpoints

Start the server first: `docker compose up` or `uvicorn app.main:app --reload`

In [ ]:
import httpx

BASE_URL = 'http://localhost:8000'

# Submit a task
response = httpx.post(
    f'{BASE_URL}/task',
    json={'task': 'Compare LangChain vs LlamaIndex for RAG applications', 'require_approval': False}
)
data = response.json()
task_id = data['task_id']
print(f'Task submitted: {task_id}')
print(f'Status: {data["status"]}')

In [ ]:
import time

# Poll until complete
for _ in range(30):
    status = httpx.get(f'{BASE_URL}/task/{task_id}').json()
    print(f'Status: {status["status"]}')
    if status['status'] in ('completed', 'failed'):
        break
    time.sleep(2)

if status['status'] == 'completed':
    print('\nFinal result:')
    print(status['result'])

## Step 4 — Human-in-the-Loop Demo

Submit with `require_approval=True` to pause at each worker step.

In [ ]:
# Submit task requiring approval
response = httpx.post(
    f'{BASE_URL}/task',
    json={'task': 'Analyse AI salary trends', 'require_approval': True}
)
task_id_hitl = response.json()['task_id']
print(f'Task with HITL: {task_id_hitl}')
print('Poll /task/{id} to get checkpoint_id, then POST /checkpoint/{id}/approve')

## Next Steps

- Wire up a **real search API** (Brave Search, SerpAPI, Tavily) in `app/worker.py`
- Add **Redis persistence** so tasks survive restarts
- Add **authentication** to the FastAPI endpoints
- Deploy to **Railway / Render / AWS ECS** using the provided Dockerfile
- Add more worker types: `code_worker`, `data_worker`, `email_worker`
